# Une chaîne complète, des bandes à la carte

**Cartomize · Version 1.0**

Durée indicative : 15 minutes.

## Objectifs

Assembler deux scènes, définir une emprise, superposer des couches et produire une carte accompagnée de ses données.

## Apport de Cartomize

Une même commande coordonne le prétraitement, la composition colorée et les exports. Les produits scientifiques restent disponibles pour les analyses suivantes.

Données : paysage pédagogique généré avec une graine fixe. Les cartes et mesures ne décrivent aucun site réel.

## Préparation

Installer l’environnement décrit dans le [guide de démarrage](../README.md), puis exécuter les cellules dans l’ordre. Chaque exécution utilise un nouveau dossier de résultats.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from IPython.display import display, Image
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'course.py').is_file() or (p/'tutorials/course.py').is_file())
root = root if (root/'course.py').is_file() else root/'tutorials'
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import cartomize as cm
from course import demo, run_directory, show_raster, show_rgb, gallery, CRS, CLASSES, ROLES

In [2]:
out = run_directory("00_premiere_carte")
d = demo(out/"data")

## 1. Examiner les entrées

Le jeu pédagogique comprend deux scènes avec recouvrement, sept bandes de réflectance simulée, une emprise, des axes et des stations. Ces scènes ne comportent pas de nuages ; le masquage est désactivé explicitement.

In [3]:
scenes = cm.discover_scenes(d['scenes'])
pd.DataFrame([{'scène': s.scene_id, 'bandes': ', '.join(s.bands), 'capteur': s.sensor} for s in scenes])

Out[3]: 
         scène                                         bandes    capteur
0  synthetic_A  blue, green, red, nir, swir1, swir2, rededge1  synthetic
1  synthetic_B  blue, green, red, nir, swir1, swir2, rededge1  synthetic


## 2. Exécuter la production

`aoi` fixe la zone d’extraction ; `layers` ajoute les couches vectorielles ; `formats` définit les livrables. Changer de zone ne nécessite pas de réécrire la chaîne.

In [4]:
production = cm.cartographic_workflow(
    scenes, out/'production', aoi=d['aoi'],
    layers=[d['roads'], d['points']], mask_clouds=False,
    composition='natural', title='Images et observations',
    subtitle='Démonstration sur un paysage simulé',
    credits='Cartomize | Données synthétiques | UTM 33S',
    formats=('pdf','png'), dpi=180,
)
products = list((out/'production').rglob('*'))
pd.DataFrame({'fichier': [str(p.relative_to(out)) for p in products if p.is_file()]})

Out[4]: 
                                  fichier
0  production/multibande_source_index.tif
1              production/multibande.json
2                    production/carte.pdf
3                    production/carte.png
4               production/multibande.tif
5      production/composition_coloree.tif
6                 production/quality.json
7              production/production.json


In [5]:
preview = next((out/'production').rglob('*.png'))
display(Image(filename=str(preview)))

## 3. Contrôler le résultat

Le GeoTIFF multibande contient des valeurs de réflectance. Le rendu RVB étiré est un produit d’affichage. Un PDF ne remplace pas les rasters utilisés pour l’analyse.

In [6]:
assert list((out/'production').rglob('*.pdf'))
assert list((out/'production').rglob('*.json'))
with rasterio.open(production.multiband) as src:
    assert src.count == 7
    display(pd.DataFrame({'bande': src.indexes, 'rôle': src.descriptions}))

,bande,rôle
0,1,blue
1,2,green
2,3,red
3,4,nir
4,5,swir1
5,6,swir2
6,7,rededge1


## Résultats de référence

![Résultat du cours](../gallery/00_premiere.png)

## Exercice

Reprendre la commande avec `composition=("nir", "red", "green")` dans un nouveau dossier. Quels fichiers changent de fonction ?

<details>
<summary>Éléments de correction</summary>

La composition colorée change ; le multibande scientifique conserve ses bandes et ses valeurs. Utiliser un nouveau dossier, par exemple `out/"fausses_couleurs"`.

</details>

## Reproduction

Les paramètres, la graine et les chemins sont explicites dans les cellules. Le répertoire `out` contient les produits de cette exécution. Adapter les sources, le système de coordonnées et les paramètres avant de transférer la méthode à une étude.